# HEART v2 — General ligand audit + existing H200 checkpoint evaluation (10,000 molecules)

**Scope: exactly two tasks.** We do **not** build lipids, continue pretraining, retrain the model, modify an LMDB or checkpoint, or submit Slurm jobs.

1. **Full original general ligand audit:** count exact *raw-SMILES identity* duplicates (optional canonical isomeric SMILES audit), train/validation overlap, and objectively compare the number of unique molecules with LUMI-lab's 13,369,320 distinct generic molecules. The 18.8M record count **does not imply** that deduplication yields 13.37M. A quick random preview is included, but it is **not** a full-dataset estimate.
2. **H200 checkpoint evaluation:** sample 10,000 `valid.lmdb` molecules, preserving the 192 validation indices selected by the original H200 notebook (`seed=17`). Use the **unchanged HEART `unimol_contrastive` task/model/loss**, numeric LMDB indexing, and **unchanged two-view collater batch method**. Evaluate masked-atom prediction, coordinate/distance recovery, NT-Xent contrastive learning, and raw CLS / projected molecular embeddings. Show results inline in the notebook (tables, PCA, UMAP).

**GitHub source:** `andypan0214/heart-molecular-pretraining-v2`, branch **`v2/lipid-continual`**. This branch's repo root contains `model/` (there is no nested `lipid-continual` folder). Its general task/model/loss are frozen from the original repo. `model/HEART_H200_checkpoint_best_embedding_evaluation.ipynb` is the 192-molecule evaluation reference; its `dataset.collater([...])` method is preserved.

**Execution:** Run cells interactively in your existing `heart_gpu` Jupyter kernel. Both costly steps have explicit flags (`RUN_FULL_RAW_AUDIT`, `RUN_EVAL_10K`), default off. There is **no `sbatch` anywhere**. The notebook cannot use compute it has not been allocated; GPU evaluation requires GPU visibility in the Jupyter kernel. Large scans can approach the Jupyter session time limit.

**Truthfulness:** A successful masked-atom/geometry/contrastive evaluation supports *pretraining-objective learning*. Neither these objectives nor visually distinct UMAP clusters establish prediction of lipid delivery or experimental molecular properties.

## 0 — Verify the actual `/scratch` layout (no assumptions about which repo holds the moved run)

The **three named folders are siblings**, not nested in one another. `checkpoint_best.pt` is selected from the H200 run under **v2 first**; the original checkout is a fallback. The selected path is displayed, never silently assumed. Model source is always **v2**, not the historical checkout. GitHub's committed `config/heart_v2.yaml` still contains `/home` paths; this notebook intentionally uses your newer `/scratch` relocation.

In [ ]:
from pathlib import Path
import os, sys, time, json, subprocess, warnings, importlib.util
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

BASE = Path('/scratch/ypan/heart_liver/model_train')
REPO_V2 = BASE / 'heart-molecular-pretraining-v2'
REPO_ORIGINAL = BASE / 'heart-molecular-pretraining'
DATA_DIR = BASE / 'UniMolReference' / 'ligand_data' / 'ligands'
UNICORE_ROOT = BASE / 'third_party' / 'Uni-Core'
CODE = BASE / 'HEART_v2_General_Ligand_Audit_and_10K'
RESULTS = BASE / 'heart_evaluation' / 'general_audit_10k_v2'
AUDIT_DIR = RESULTS / 'full_ligand_audit'
MANIFEST_DIR = RESULTS / 'validation_10k_seed17'
H200_RUN_REL = Path('runs/h200/full/heart-h200-b448-v1')
CHECKPOINT_OVERRIDE = None  # set explicit absolute Path only if needed

# User moved the full run to v2; prefer it there, but check old location as fallback.
candidates = [REPO_V2 / H200_RUN_REL / 'checkpoint_best.pt',
              REPO_ORIGINAL / H200_RUN_REL / 'checkpoint_best.pt']
CHECKPOINT = Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else next((p for p in candidates if p.is_file()), None)

for name, p in [('v2 source',REPO_V2),('historical repo',REPO_ORIGINAL),
                ('general data',DATA_DIR),('notebook helpers',CODE),('pinned Uni-Core',UNICORE_ROOT)]:
    print(f'{name:20} {p} | exists={p.exists()}')
print('H200 checkpoint candidates:')
for p in candidates: print(' ', 'FOUND ' if p.is_file() else 'ABSENT', p)
print('SELECTED CHECKPOINT:', CHECKPOINT if CHECKPOINT else 'NONE — fix path before evaluation')
print('Kernel:', sys.executable)

assert (REPO_V2/'model'/'unimol'/'tasks'/'unimol_contrastive.py').is_file(), 'Incorrect v2 repo location.'
assert (DATA_DIR/'train.lmdb').is_file() and (DATA_DIR/'valid.lmdb').is_file() and (DATA_DIR/'dict.txt').is_file(), 'Missing general LMDB files.'
assert CODE.is_dir() and (CODE/'evaluate_heart.py').is_file(), 'Unzip package directly under BASE.'

# Git check is read-only. A detached HEAD is okay if the pinned source is correct.
try:
    r = subprocess.run(['git','-C',str(REPO_V2),'branch','--show-current'],capture_output=True,text=True,timeout=10)
    print('v2 branch:', r.stdout.strip() or '(detached HEAD / unavailable)')
    r = subprocess.run(['git','-C',str(REPO_V2),'rev-parse','HEAD'],capture_output=True,text=True,timeout=10)
    print('v2 commit:', r.stdout.strip() or '(unavailable)')
except Exception as exc: print('Git metadata unavailable:',repr(exc))

# All outputs stay OUTSIDE the training source and original LMDB.
assert not str(RESULTS).startswith(str(DATA_DIR))
assert CHECKPOINT is None or CHECKPOINT.resolve().name == 'checkpoint_best.pt'

## 1 — Historical H200 training evidence (display existing log-derived CSVs; no training)

Use this only to describe the training trend. The primary new evidence will come from the **ligand data audit** and **10,000 validation forward passes**; historical `masked_acc ≈ 0.983` and `valid_loss ≈ 0.624` are reported at the original training validation batch size (448 molecules).

In [ ]:
import matplotlib.pyplot as plt
tr_path = CODE/'original_train_inner_history.csv'
va_path = CODE/'original_valid_history.csv'
if tr_path.is_file() and va_path.is_file():
    tr=pd.read_csv(tr_path).sort_values('num_updates')
    va=pd.read_csv(va_path).sort_values('num_updates')
    fig, ax = plt.subplots(figsize=(10,4))
    ax.plot(tr.num_updates,tr.loss,label='Historical training (logged batches)',alpha=.55,linewidth=.8)
    ax.plot(va.num_updates,va.loss,label='Historical validation',linewidth=2)
    ax.set(xlabel='Optimization updates',ylabel='Original weighted loss',title='Existing HEART H200 training (not re-run)')
    ax.legend();ax.grid(alpha=.2);plt.tight_layout();plt.show()
    valid_keys=[x for x in ['num_updates','loss','masked_acc','masked_token_loss','masked_coord_loss','masked_dist_loss','contrastive_loss'] if x in va.columns]
    display(va.tail(5)[valid_keys])
else:
    print('Historical log CSVs unavailable. This does not block the audit or checkpoint evaluation.')

## 2 — Quick original LMDB preview (read-only, exact sampled records)

Checks the LMDB field `smi` using **numeric keys** (matching HEART's `LMDBDataset[index]`). This helps inspect schema, invalid identifiers and within-preview duplicates. **It CANNOT estimate the global duplicate rate or prove that 18.84M reduces to 13.37M.** Multiple 3D conformations stored in one LMDB record do **not** count as different molecules.

In [ ]:
import lmdb, pickle
from collections import Counter

PREVIEW_TRAIN_N = 20000  # quick overview; still not a global estimate
PREVIEW_VALID_N = 3000
PREVIEW_SEED = 17

def sample_numeric_lmdb(lmdb_path, n, seed):
    env=lmdb.open(str(lmdb_path),subdir=False,readonly=True,lock=False,readahead=False,meminit=False,max_readers=128)
    try:
        with env.begin(write=False) as txn:
            count=txn.stat()['entries']
            ids=np.random.default_rng(seed).choice(count,size=min(n,count),replace=False)
            smis=[]; missing=0; n_confs=[]
            for i in ids:
                b=txn.get(str(int(i)).encode('ascii'))
                if b is None: raise RuntimeError(f'Missing numeric LMDB key {int(i)}')
                d=pickle.loads(b)  # Trusted local LMDB only.
                if not isinstance(d,dict): raise TypeError(f'Expected dict at {i}, got {type(d)}')
                s=d.get('smi',d.get(b'smi',''))
                if isinstance(s,bytes):s=s.decode('utf8',errors='replace')
                if not isinstance(s,str) or not s.strip(): missing+=1;s=''
                smis.append(s.strip())
                coords=d.get('coordinates',[])
                n_confs.append(len(coords) if hasattr(coords,'__len__') else None)
        return count,ids,smis,n_confs,missing
    finally: env.close()

def sampled_duplicates(smis, policy):
    if policy=='canonical':
        from rdkit import Chem, RDLogger
        RDLogger.DisableLog('rdApp.error')
        identities=[];invalid=0
        for s in smis:
            if not s:continue
            mol=Chem.MolFromSmiles(s)
            if mol is None: invalid+=1;continue
            identities.append(Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True))
    else:
        identities=[s for s in smis if s];invalid=0
    c=Counter(identities)
    n=len(identities)
    return {'identity':policy,'usable_sampled':n,'unique_sampled':len(c),
            'duplicate_excess_sampled':n-len(c),'duplicate_percent_within_sample':100*(n-len(c))/n if n else None,
            'rdkit_invalid_sampled':invalid}

n_train,ids_train,smis_train,slots_train,miss_train=sample_numeric_lmdb(DATA_DIR/'train.lmdb',PREVIEW_TRAIN_N,PREVIEW_SEED)
n_valid,ids_valid,smis_valid,slots_valid,miss_valid=sample_numeric_lmdb(DATA_DIR/'valid.lmdb',PREVIEW_VALID_N,PREVIEW_SEED+1)
print(f'Actual LMDB counts: TRAIN {n_train:,} / VALID {n_valid:,}')
print('Stored conformer slots (sample):',Counter(slots_train).most_common(5))
report=[]
for name,ss,missing in [('train',smis_train,miss_train),('valid',smis_valid,miss_valid)]:
    for policy in ('raw','canonical'):
        report.append({'split':name,'sample_records':len(ss),'missing_smi':missing,**sampled_duplicates(ss,policy)})
display(pd.DataFrame(report))
print('NOTE: these duplicate percentages refer ONLY to collisions within the selected sample.')

## 3 — EXACT full 18.84M duplicate count, train/valid overlap, and LUMI 13.37M comparison

**To answer the first primary question exactly, a full scan is necessary.** The following flags are **off by default** so Run-All never accidentally starts an 18.84M-record scan. Set `RUN_FULL_RAW_AUDIT=True` and rerun this cell to compute the full **raw-SMILES** count *inside Jupyter*, with `sort -u` on `/scratch`; **no new job submission**. This writes only identifier hashes and summary files under `RESULTS`, not the original LMDB. Re-running reuses completed exports/sorts; if an interrupted `.part` file exists, inspect it before retrying.

For a stricter molecular identity check, enable `RUN_FULL_CANONICAL_AUDIT=True` **after** the raw audit. Canonicalizing all 18.84M molecules with RDKit is substantially more CPU-intensive. We report both, without forcing either to match LUMI's 13M. BLAKE2b-128 hash collision risk is negligible but not mathematically zero. Molecular equivalences beyond canonical isomeric SMILES (tautomers, charge/salt conventions) remain out of scope.

In [ ]:
RUN_FULL_RAW_AUDIT = False       # Set True explicitly to compute the full 18.84M raw duplicate rate.
RUN_FULL_CANONICAL_AUDIT = False # Optional much more expensive RDKit canonical audit.
AUDIT_THREADS = 4
AUDIT_SORT_RAM = '512M'
LUMI_GENERAL_DISTINCT = 13_369_320

AUDIT_DIR.mkdir(parents=True,exist_ok=True)
for identity, do_run in [('raw',RUN_FULL_RAW_AUDIT),('canonical',RUN_FULL_CANONICAL_AUDIT)]:
    out=AUDIT_DIR/identity
    result=out/f'audit_summary_{identity}.json'
    if result.is_file():
        print('EXISTING completed audit:',result,'(not re-running)')
    elif do_run:
        command=[sys.executable,str(CODE/'audit_lmdb.py'),'run',
                 '--train',str(DATA_DIR/'train.lmdb'),'--valid',str(DATA_DIR/'valid.lmdb'),
                 '--out',str(out),'--identity',identity,'--threads',str(AUDIT_THREADS),
                 '--sort-buffer',AUDIT_SORT_RAM,'--report-every','500000']
        print('Executing directly in this Jupyter kernel — NO sbatch:',identity,flush=True)
        subprocess.run(command,check=True)
    else:
        print(f'{identity}: full audit not yet performed; enable RUN_FULL_{identity.upper()}_AUDIT to compute it.')

comparison=[]
for identity in ('raw','canonical'):
    result=AUDIT_DIR/identity/f'audit_summary_{identity}.json'
    if not result.is_file(): continue
    obj=json.loads(result.read_text())
    t,v,ov=obj['train'],obj['valid'],obj['train_valid_overlap']
    # Never label a partial result as exact 18.84M.
    if t['limited'] or v['limited'] or t['total_records_scanned']!=n_train or v['total_records_scanned']!=n_valid:
        print('PARTIAL/UNMATCHED scan, not showing as full:',result)
        continue
    comparison.append({
        'identity':identity,'train_records':t['total_records_scanned'],
        'usable_train_records':t['usable_records'],'unique_train_molecules':t['unique_identifiers'],
        'duplicate_record_excess':t['duplicate_record_excess'],
        'duplicate_rate_pct_of_usable':100*t['duplicate_record_excess_fraction'],
        'valid_records_seen_in_train':ov['overlapping_validation_records'],
        'valid_overlap_pct_of_usable':100*ov['fraction_validation_records_seen_in_train'],
        'difference_vs_LUMI_13_369_320':t['unique_identifiers']-LUMI_GENERAL_DISTINCT,
    })
if comparison:
    cmp=pd.DataFrame(comparison);display(cmp)
    fig,ax=plt.subplots(figsize=(7,4))
    vals=[LUMI_GENERAL_DISTINCT]+[z['unique_train_molecules'] for z in comparison]
    labels=['LUMI reported']+[f'HEART {z["identity"]}' for z in comparison]
    ax.bar(labels, np.array(vals)/1e6)
    ax.set(ylabel='Distinct molecules (millions)',title='Dataset-size comparison (not proof of identical dataset)')
    plt.xticks(rotation=12);plt.tight_layout();plt.show()
    print('A numerical match to ~13.37M is NOT a reproduction claim without verifying data filtering and identity definitions.')
else:
    print(f'Full unique count: UNKNOWN. LUMI reference = {LUMI_GENERAL_DISTINCT:,}.')
    print('The 18.84M records alone cannot tell us whether deduplication yields ~13.37M.')

## 4 — Make the exact 10,000-molecule H200 validation sample (no Slurm)

The original H200 192-molecule reference uses `np.random.default_rng(17).choice(len(valid),192,replace=False)`. We **preserve that exact 192-ID set** and extend to 10,000 unique numeric indices from the remaining validation records. Critically, HEART reads LMDB by `txn.get(str(index))`, **not the lexicographic cursor position**; metadata and embeddings are explicitly aligned.

In [ ]:
N_EVAL = 10000
SAMPLE_SEED = 17
MANIFEST_DIR.mkdir(parents=True,exist_ok=True)
manifest_csv=MANIFEST_DIR/'sample_metadata.csv'
manifest_info=MANIFEST_DIR/'sampling_summary.json'
if manifest_csv.is_file() and manifest_info.is_file():
    print('Reusing existing 10,000-molecule sample:',manifest_csv)
elif manifest_csv.exists() or manifest_info.exists():
    raise RuntimeError('Only part of the manifest exists. Inspect it; this notebook will not overwrite evidence.')
else:
    cmd=[sys.executable,str(CODE/'prepare_subset.py'),'--lmdb',str(DATA_DIR/'valid.lmdb'),
         '--out',str(MANIFEST_DIR),'--n',str(N_EVAL),'--seed',str(SAMPLE_SEED),'--legacy-n','192']
    subprocess.run(cmd,check=True)
manifest=pd.read_csv(manifest_csv,keep_default_na=False)
mi=json.loads(manifest_info.read_text())
assert len(manifest)==N_EVAL and manifest['lmdb_index'].nunique()==N_EVAL
assert mi['source_lmdb_records']==n_valid
assert (manifest['lmdb_index'].astype(str)==manifest['lmdb_key'].astype(str)).all(), 'Incorrect LMDB numeric mapping'
legacy_reference=np.random.default_rng(SAMPLE_SEED).choice(n_valid,192,replace=False)
assert np.array_equal(manifest.lmdb_index.iloc[:192].to_numpy(),legacy_reference), '192-ID historical sample was not preserved'
display(manifest.head(5))
print(f'10,000 / {n_valid:,} distinct validation record indices, first 192 unchanged.')

## 5 — H200 checkpoint: evaluate the existing model, **same collater + same original loss**

**No code refactoring of the trained HEART model/loss.** The helper loads the H200 weights via strict `load_state_dict`, builds the `unimol_contrastive` task from **v2 general code**, keeps `mode='train'` (necessary for this model's six-output evaluation forward), and calls the **original `task.build_loss(args)`** with its **original 2-view** `dataset.collater([dataset[i] ...])` for each batch. `model.eval()` + inference mode, no backpropagation or optimizer.

The original H200 embedding notebook uses `BATCH_MOLECULES=8` (two views = 16 sequences). The default below retains **8**, which preserves its evaluation batching. The *training/official validation* used **448 molecules per batch**, so NT-Xent (and total weighted loss) at batch 8 **cannot be directly compared numerically** with the historical validation loss 0.624. If you have a GPU-visible Jupyter session and specifically want 448-objective comparability, deliberately change `BATCH_MOLECULES=448` and use a new output directory; that is **not** the default.

Set `RUN_EVAL_SMOKE=True` for a small read-only diagnostic, then `RUN_EVAL_10K=True` for the requested 10k forward evaluation. Neither action submits any Slurm job. If Jupyter has no GPU, you may set `ALLOW_CPU_EVALUATION=True` (it may be slow); it will not silently fall back to CPU.

In [ ]:
RUN_EVAL_SMOKE = False      # 8 molecules, optional
RUN_EVAL_10K = False        # set True to run 10,000 in this notebook, NO sbatch
ALLOW_CPU_EVALUATION = False
BATCH_MOLECULES = 8        # SAME default as original H200 embedding notebook

import torch
print('torch:',torch.__version__,'CUDA visible:',torch.cuda.is_available())
assert CHECKPOINT is not None and CHECKPOINT.is_file(), 'No H200 checkpoint detected. See section 0; set CHECKPOINT_OVERRIDE.'
RUN_ROOT=RESULTS/'checkpoint_evaluation'
RUN_ROOT.mkdir(parents=True,exist_ok=True)

# Run folder encodes batch size because NT-Xent depends on it. Never overwrite earlier evaluation data.
def eval_command(out,limit):
    command=[sys.executable,str(CODE/'evaluate_heart.py'),
             '--repo',str(REPO_V2),'--data-dir',str(DATA_DIR),
             '--checkpoint',str(CHECKPOINT),
             '--manifest-dir',str(MANIFEST_DIR),'--out',str(out),
             '--batch-size',str(BATCH_MOLECULES)]
    if UNICORE_ROOT.is_dir():command += ['--unicore-dir',str(UNICORE_ROOT)]
    if limit:command += ['--limit',str(limit)]
    if not torch.cuda.is_available():
        if not ALLOW_CPU_EVALUATION:raise RuntimeError('No GPU in this kernel. No job is submitted. To allow CPU, set ALLOW_CPU_EVALUATION=True.')
        command += ['--cpu']
    return command

for label,trigger,limit in [('smoke_8',RUN_EVAL_SMOKE,8),('full_10k',RUN_EVAL_10K,0)]:
    out=RUN_ROOT/f'{label}_batch{BATCH_MOLECULES}'
    summary_path=out/'evaluation_summary.json'
    if summary_path.is_file():
        print(label,'existing completed metrics:',summary_path,'— not rerunning')
    elif trigger:
        out.mkdir(parents=True,exist_ok=True)
        print('Executing inference directly in Jupyter (no job):',label,flush=True)
        subprocess.run(eval_command(out,limit),check=True)
    else:
        print(label,'not yet executed; set RUN_EVAL_SMOKE or RUN_EVAL_10K to True when ready.')

## 6 — Visualize the actual 10,000-molecule pretraining-objective results

The tables below read `objective_batches.csv`, `molecule_diagnostics.csv` and `evaluation_summary.json` generated by the unchanged HEART criterion wrapper. **Masked accuracy is hits/targets across genuinely masked atoms**, not across all atoms. Geometry RMSE compares reconstructed targets with the noisy input. NT-Xent and retrieval refer to positives/negatives **within each evaluated batch**, not across the entire 10k.

In [ ]:
RUN_10K=RUN_ROOT/f'full_10k_batch{BATCH_MOLECULES}'
summary_path=RUN_10K/'evaluation_summary.json'
if not summary_path.is_file():
    display(Markdown('**10k evaluation not run yet.** Enable `RUN_EVAL_10K=True` in Section 5 when resources are available.'))
else:
    metrics=json.loads(summary_path.read_text())
    assert metrics.get('status')=='complete' and metrics.get('n_molecules')==10000,'Evaluation is incomplete or wrong-sized'
    display(pd.DataFrame([{
        'Validation molecules':metrics.get('n_molecules'),
        'Masked accuracy':metrics.get('masked_atom_accuracy'),
        'Masked majority-class baseline':metrics.get('masked_atom_majority_class_baseline_accuracy'),
        'Coordinate RMSE (pred)':metrics.get('coord_pred_rmse'),
        'Coordinate RMSE (noisy)':metrics.get('coord_noisy_rmse'),
        'RMSE improvement fraction':metrics.get('coord_rmse_reduction_fraction'),
        'NT-Xent batch top-1 retrieval':metrics.get('contrastive_top1_both_directions'),
        'Mean positive cosine':metrics.get('positive_projection_cosine_mean'),
        'Mean negative cosine':metrics.get('negative_projection_cosine_mean_of_batches')
    }]))
    batch=pd.read_csv(RUN_10K/'objective_batches.csv')
    cols=[c for c in ('masked_token_loss','masked_coord_loss','masked_dist_loss','contrastive_loss','official_weighted_loss') if c in batch.columns]
    display(batch[cols].describe().round(4))
    mol=pd.read_csv(RUN_10K/'molecule_diagnostics.csv')
    fig, axes=plt.subplots(1,3,figsize=(13,3.5))
    mol.masked_atom_accuracy.dropna().hist(bins=25,ax=axes[0]); axes[0].set(title='Masked atom accuracy per view',xlabel='Accuracy')
    z=batch[['coord_pred_rmse','coord_noisy_rmse']].dropna()
    if len(z):
        axes[1].scatter(z.coord_noisy_rmse,z.coord_pred_rmse,alpha=.3,s=10)
        lim=max(z.max());axes[1].plot([0,lim],[0,lim],ls='--',c='gray')
    axes[1].set(title='Coordinate recovery vs noisy input',xlabel='Noisy-input RMSE',ylabel='Predicted RMSE')
    if 'contrastive_loss' in batch:batch.contrastive_loss.dropna().hist(bins=25,ax=axes[2])
    axes[2].set(title='NT-Xent loss across batches',xlabel='Loss')
    plt.tight_layout();plt.show()
    display(pd.read_csv(RUN_10K/'atom_token_class_accuracy.csv').head(15))

## 7 — Molecular representations: raw CLS / projected CLS, PCA, UMAP (same 10k molecules)

This is **visualization of learned embeddings**, not a molecular property prediction benchmark. Computing `n×n` cosine similarity matrices for n=10,000 is intentionally avoided (that was practical only at n=192). PCA is via randomized SVD; UMAP runs on the 512D molecular embeddings and may take some CPU time. Model weights and input data remain read-only.

In [ ]:
RUN_REPRESENTATION_PLOTS = True  # plots only when complete 10k embedding output exists
if not (RUN_10K/'cls_view0.npy').is_file():
    print('No 10k CLS array yet; run Section 5 first (RUN_EVAL_10K=True).')
elif RUN_REPRESENTATION_PLOTS:
    from sklearn.decomposition import PCA
    X=np.load(RUN_10K/'cls_view0.npy',mmap_mode='r')
    XP=np.load(RUN_10K/'projected_view0.npy',mmap_mode='r')
    assert X.shape[0]==XP.shape[0]==10000 and X.ndim==XP.ndim==2
    assert np.isfinite(X).all() and np.isfinite(XP).all()
    sample_meta=manifest[['sample_order','lmdb_index','smiles']].copy()
    from rdkit import Chem, RDLogger
    from rdkit.Chem import Descriptors, rdMolDescriptors
    RDLogger.DisableLog('rdApp.error')
    mw=[];heavy=[];clogp=[];ring=[]
    for smi in sample_meta.smiles.astype(str):
        mol=Chem.MolFromSmiles(smi)
        if mol is None: mw.append(np.nan);heavy.append(np.nan);clogp.append(np.nan);ring.append(np.nan)
        else:
            mw.append(Descriptors.MolWt(mol)); heavy.append(mol.GetNumHeavyAtoms())
            clogp.append(Descriptors.MolLogP(mol));ring.append(rdMolDescriptors.CalcNumRings(mol))
    sample_meta['MW']=mw;sample_meta['heavy_atoms']=heavy;sample_meta['cLogP']=clogp;sample_meta['rings']=ring
    print('Source molecular property coverage:',sample_meta['MW'].notna().mean())
    figure_dir=RUN_10K/'figures';figure_dir.mkdir(parents=True,exist_ok=True)
    for label,emb in [('CLS_raw',X),('Projector_NTXent',XP)]:
        # Randomized PCA avoids a very large 10k x 10k similarity matrix.
        pca=PCA(n_components=2,svd_solver='randomized',random_state=17)
        XY=pca.fit_transform(np.asarray(emb,dtype=np.float32))
        fig,axes=plt.subplots(1,2,figsize=(12,4.5))
        for ax,property_name in zip(axes,['heavy_atoms','cLogP']):
            col=sample_meta[property_name].to_numpy()
            sc=ax.scatter(XY[:,0],XY[:,1],c=col,s=3,alpha=.65,cmap='viridis',rasterized=True)
            ax.set(title=f'{label} PCA — {property_name}',xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
                   ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
            fig.colorbar(sc,ax=ax,label=property_name)
        plt.tight_layout();plt.savefig(figure_dir/f'{label}_PCA.png',dpi=160);plt.show()
        try:
            import umap
        except ImportError:
            print('UMAP skipped: install umap-learn in existing environment if permitted. No auto-install.');continue
        reducer=umap.UMAP(n_components=2,n_neighbors=15,min_dist=.1,metric='cosine',random_state=17)
        XYu=reducer.fit_transform(np.asarray(emb,dtype=np.float32))
        assert XYu.shape==(10000,2)
        fig,axes=plt.subplots(1,2,figsize=(12,4.5))
        for ax,prop in zip(axes,['heavy_atoms','cLogP']):
            sc=ax.scatter(XYu[:,0],XYu[:,1],c=sample_meta[prop],s=3,alpha=.65,cmap='viridis',rasterized=True)
            ax.set(title=f'{label} UMAP — {prop}',xlabel='UMAP 1',ylabel='UMAP 2')
            fig.colorbar(sc,ax=ax,label=prop)
        plt.tight_layout();plt.savefig(figure_dir/f'{label}_UMAP.png',dpi=160);plt.show()
        pd.DataFrame({'sample_order':sample_meta.sample_order,'lmdb_index':sample_meta.lmdb_index,
                      'UMAP1':XYu[:,0],'UMAP2':XYu[:,1]}).to_csv(figure_dir/f'{label}_UMAP.csv',index=False)

## 8 — Honest scientific interpretation and decision before lipid continual pretraining

Read this section *after* completing the requested analyses:

- **Duplicate rate:** The exact raw or canonical `train_unique` count comes only from a **complete scan**. Do not extrapolate a 20k collision rate to all 18.84M molecules. A value close to LUMI's 13,369,320 is evidence of similar **size**, not necessarily identical molecules, filtering or conformation protocols.
- **Generalization / leakage:** Check train–validation molecular overlap. If high, the historical validation loss is potentially optimistic even with smooth train/validation curves. This does not by itself prove overfitting, but it means clean held-out testing may be needed.
- **Masked atom:** Accuracy and class breakdown support atom recovery, but do not prove overall chemical understanding. Compare to majority-class baseline shown above.
- **Geometry:** Coordinate/distance losses are the original HEART reconstruction objectives. A lower predicted coordinate RMSE than noisy-input RMSE is a useful denoising check, not a guarantee of accurate force fields or conformational energies.
- **Contrastive:** NT-Xent measures consistency of two views and discrimination from *in-batch* negatives. Its numerical value changes with evaluation batch size; compare only batch-matched conditions.
- **CLS / UMAP:** Embedding spread and correlations with chemistry descriptors indicate learned structure but are not labels of functional RNA delivery.
- **Checkpoint decision:** Keep the frozen H200 general checkpoint unless duplicate/validation leakage or meaningful downstream failures justify a new training experiment. None of these cells modify that checkpoint, lipid library pipeline or continual code.

**Repository provenance:** `andypan0214/heart-molecular-pretraining-v2` branch `v2/lipid-continual`; general `unimol_contrastive` task/model/loss (frozen); source notebook `model/HEART_H200_checkpoint_best_embedding_evaluation.ipynb`; Uni-Mol general ligand `train.lmdb`/`valid.lmdb`.